# SolarSDE — Final Publication Notebook

**Code provenance:** every stage below pulls and runs the actual modules from [github.com/keshavkrishnan08/SDE](https://github.com/keshavkrishnan08/SDE) — nothing is rewritten or embedded.

**What this notebook produces (one GPU run):**
1. Full SKIPP'D pipeline (517 days, VAE + optical-flow motion + CTI)
2. **Both architectures** trained: closed-form Mixture-of-OU *and* Euler-Maruyama latent rollout
3. **Ensemble** of the two + **champion selection on validation** (never test)
4. SkyGPT exact-benchmark (identical Nov–Dec 2019 cloudy test) for **all variants**, full 1–30 min band
5. Baselines, ablations, stratified+DM, leave-one-month-out CV, PIT/bootstrap, ramp AUROC, CTI validation, multi-level reliability, sampling efficiency, compute cost, Holm-Bonferroni, CAISO economics + sensitivity, figures, LaTeX tables

Every stage is failure-isolated: an error prints and the run continues to the final zip.

*Honesty note: the head-to-head vs SkyGPT (CRPS 2.81 at h=15, their cloudy test) is reported exactly as measured — whichever way it comes out.*

## 0. Environment

In [1]:
# ==== Setup: environment, directories, torch warmup ====
import os, sys, json, math, time, gc, shutil, subprocess, traceback
from pathlib import Path
import numpy as np, pandas as pd

# torch._dynamo warmup (some Kaggle builds crash on lazy import at optimizer creation)
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
# CUDA_LAUNCH_BLOCKING makes device-side asserts raise at the offending op (not a
# later CUBLAS call), so a crash is attributed to the real cause and isolated to
# its own stage instead of surfacing mysteriously downstream.
os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "1")
import torch
try:
    import torch._utils, torch._dynamo  # noqa: F401
except Exception as _e:
    print(f"[WARN] dynamo warmup: {_e} — continuing")
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

IN_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
IN_COLAB = "google.colab" in sys.modules
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kaggle={IN_KAGGLE} Colab={IN_COLAB} device={DEVICE}")
if DEVICE.type != "cuda":
    print("[WARN] No GPU — enable a GPU runtime. Training both architectures on CPU is impractical.")

ROOT = (Path("/kaggle/working") if IN_KAGGLE else Path.cwd()) / "final_run"
PERSIST_DIR = ROOT / "outputs"; WORK_DIR = ROOT / "work"; DATA_DIR = WORK_DIR / "data"
CHECKPOINT_DIR = PERSIST_DIR / "checkpoints"; RESULTS_DIR = PERSIST_DIR / "results"
LATENT_DIR = PERSIST_DIR / "latents"; SPLITS_DIR = PERSIST_DIR / "splits"
EXTENDED_DIR = PERSIST_DIR / "extended"; FIGURES_DIR = PERSIST_DIR / "figures"
for d in [DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, LATENT_DIR, SPLITS_DIR, EXTENDED_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ===== Run configuration (the only knobs you may want to touch) =====
Z_DIM = 64
SKIPPD_VAE_EPOCHS  = 12     # CS-VAE epochs
CLOSEDFORM_EPOCHS  = 60     # closed-form SDE training epochs
ROLLOUT_EPOCHS     = 35     # rollout SDE epochs (each epoch costs ~3-4x closed-form)
CV_EPOCHS          = 15     # cross-validation epochs per fold
CV_MAX_FOLDS       = 6
print(f"PERSIST_DIR={PERSIST_DIR}")
print(f"Config: VAE={SKIPPD_VAE_EPOCHS}ep, closed-form={CLOSEDFORM_EPOCHS}ep, "
      f"rollout={ROLLOUT_EPOCHS}ep, CV={CV_EPOCHS}ep x {CV_MAX_FOLDS} folds")


Kaggle=True Colab=False device=cuda
PERSIST_DIR=/kaggle/working/final_run/outputs
Config: VAE=12ep, closed-form=60ep, rollout=35ep, CV=15ep x 6 folds


## 1. Pull the codebase from GitHub (the repo code IS the experiment code)

In [2]:
# ==== Pull the SolarSDE codebase from GitHub and import the actual modules ====
# The code that runs below IS the repo code (github.com/keshavkrishnan08/SDE),
# not a copy embedded in this notebook.
REPO_HTTPS = "https://github.com/keshavkrishnan08/SDE.git"
REPO_ZIP   = "https://github.com/keshavkrishnan08/SDE/archive/refs/heads/main.zip"
REPO_DIR = ROOT / "sde_repo"

def _clone_repo():
    if (REPO_DIR / "notebooks" / "_solarsde_v2.py").exists():
        print(f"  repo already present at {REPO_DIR}")
        # refresh to latest main (best effort)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
                       capture_output=True, timeout=120)
        return True
    for attempt in range(1, 4):
        try:
            print(f"  git clone (attempt {attempt}) ...")
            r = subprocess.run(["git", "clone", "--depth", "1", REPO_HTTPS, str(REPO_DIR)],
                               capture_output=True, text=True, timeout=300)
            if r.returncode == 0 and (REPO_DIR / "notebooks").exists():
                return True
            print(f"    clone failed: {r.stderr[:200]}")
        except Exception as e:
            print(f"    clone error: {e}")
        time.sleep(5)
    # Fallback: download the repo as a zip archive
    try:
        print("  falling back to zip archive download ...")
        import urllib.request, zipfile, io
        with urllib.request.urlopen(REPO_ZIP, timeout=300) as r:
            zf = zipfile.ZipFile(io.BytesIO(r.read()))
        zf.extractall(ROOT)
        extracted = next(ROOT.glob("SDE-*"))
        if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
        extracted.rename(REPO_DIR)
        return (REPO_DIR / "notebooks").exists()
    except Exception as e:
        print(f"    zip fallback failed: {e}")
        return False

if not _clone_repo():
    raise RuntimeError("Could not obtain the SolarSDE repo from GitHub — check network/repo access.")
MODULE_DIR = REPO_DIR / "notebooks"
sys.path.insert(0, str(MODULE_DIR))
print(f"  modules dir: {MODULE_DIR}")
print(f"  repo modules: {sorted(p.name for p in MODULE_DIR.glob('_*.py'))}")

# ---- Fallback-guarded imports: a missing/broken module never stops the run ----
def _safe_import(module, names):
    out = {}
    try:
        mod = __import__(module, fromlist=names)
        for n in names:
            out[n] = getattr(mod, n)
        print(f"  [OK]   {module}: {len(names)} constants")
    except Exception as e:
        print(f"  [FAIL] {module}: {type(e).__name__}: {str(e)[:120]}")
        for n in names:
            out[n] = f'print("[SKIP] {n} unavailable — module {module} failed to import")'
    return out

globals().update(_safe_import("_master_hardening", ["safe_stage"]))
globals().update(_safe_import("_combined_generator",
    ["SHARED_CODE", "BASELINES_CODE", "STRATIFIED_CODE", "ANALYSIS_CODE"]))
globals().update(_safe_import("_final_generator",
    ["LOAD_DATA_TOLERANT_CODE", "RAMP_AUROC_CODE", "BOOTSTRAP_CIS_CODE",
     "PIT_RELIABILITY_CODE", "ECONOMIC_CAISO_CODE", "LATEX_TABLES_CODE", "ZIP_DOWNLOAD_CODE"]))
globals().update(_safe_import("_colab_master_generator",
    ["CTI_VALIDATION_CODE", "HOLM_BONFERRONI_CODE"]))
globals().update(_safe_import("_skippd_pipeline",
    ["SKIPPD_DOWNLOAD_FULL_CODE", "SKIPPD_PREP_CODE", "SKIPPD_VAE_CODE",
     "SKIPPD_LATENTS_WRITE_CODE", "SKIPPD_HORIZON_OVERRIDE_CODE"]))
globals().update(_safe_import("_solarsde_v2",
    ["MDN_ARCHITECTURE_CODE", "STAGE_0_V2_CODE", "POST_STAGE0_V2_VERIFY_CODE", "ABLATIONS_V2_CODE"]))
globals().update(_safe_import("_solarsde_rollout",
    ["ROLLOUT_ARCH_CODE", "ABLATIONS_ROLLOUT_CODE"]))
globals().update(_safe_import("_ensemble_eval",
    ["STASH_CLOSEDFORM_CODE", "STASH_ROLLOUT_CODE", "CHAMPION_SELECT_CODE",
     "SKYGPT_TRIPLE_BENCHMARK_CODE"]))
globals().update(_safe_import("_skippd_extras",
    ["IMPLEMENTATION_DETAILS_CODE", "DATA_CARD_CODE", "COMPUTATIONAL_COST_CODE",
     "RELIABILITY_LEVELS_CODE", "SAMPLING_EFFICIENCY_CODE", "ECONOMIC_SENSITIVITY_CODE",
     "CROSS_VALIDATION_V2_CODE"]))

# If safe_stage itself failed to import, provide a minimal local fallback.
if isinstance(globals().get("safe_stage"), str):
    def safe_stage(name, code):
        ind = "\n".join("    " + l if l else "" for l in code.splitlines())
        return (f"try:\n{ind}\nexcept Exception as _e:\n"
                f"    import traceback; traceback.print_exc()\n"
                f"    print('[STAGE FAILED] {name} — continuing.')\n")
    print("  [WARN] using local fallback safe_stage")
print("\nAll modules wired. Code provenance: github.com/keshavkrishnan08/SDE @ main")


  git clone (attempt 1) ...
  modules dir: /kaggle/working/final_run/sde_repo/notebooks
  repo modules: ['_colab_master_generator.py', '_combined_generator.py', '_ensemble_eval.py', '_final_generator.py', '_final_master_generator.py', '_generator.py', '_kaggle_master_generator.py', '_master_hardening.py', '_multisite.py', '_skippd_extras.py', '_skippd_master_generator.py', '_skippd_pipeline.py', '_skippd_rollout_master_generator.py', '_skygpt_eval.py', '_solarsde_rollout.py', '_solarsde_v2.py']
  [OK]   _master_hardening: 1 constants
  [OK]   _combined_generator: 4 constants
  [OK]   _final_generator: 7 constants
  [OK]   _colab_master_generator: 2 constants
  [OK]   _skippd_pipeline: 5 constants
  [OK]   _solarsde_v2: 4 constants
  [OK]   _solarsde_rollout: 2 constants
  [OK]   _ensemble_eval: 4 constants
  [OK]   _skippd_extras: 7 constants

All modules wired. Code provenance: github.com/keshavkrishnan08/SDE @ main


## 2. Download all data — SKIPP'D (~2.3 GB) + SkyGPT exact test set

In [3]:
# ==== DOWNLOAD_SKIPPD ====
try:
    exec(safe_stage('DOWNLOAD_SKIPPD', SKIPPD_DOWNLOAD_FULL_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] DOWNLOAD_SKIPPD — continuing to next cell.')


SKIPP'D FULL download (~2.3 GB)
  pull  data/train-00000-of-00005.parquet (attempt 1) ... 448 MB
  pull  data/train-00001-of-00005.parquet (attempt 1) ... 455 MB
  pull  data/train-00002-of-00005.parquet (attempt 1) ... 471 MB
  pull  data/train-00003-of-00005.parquet (attempt 1) ... 439 MB
  pull  data/train-00004-of-00005.parquet (attempt 1) ... 419 MB
  pull  data/test-00000-of-00001.parquet (attempt 1) ... 90 MB
  pull  labels/train-00000-of-00001.parquet (attempt 1) ... 4 MB
  pull  labels/test-00000-of-00001.parquet (attempt 1) ... 0 MB
SKIPP'D files present: 8/8


## 3. Preprocess — clear-sky-PV, ramps, chronological splits

In [4]:
# ==== SKIPPD_PREP ====
try:
    exec(safe_stage('SKIPPD_PREP', SKIPPD_PREP_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SKIPPD_PREP — continuing to next cell.')


[SKIPPD-PREP] loading image parquets ...
  images: 363,375 rows, 517 days, PV [0.00, 29.59]
[SKIPPD-PREP] clear-sky-PV envelope from full labels ...
  kt mean=0.724  ramps=12,540 (3.5%)  kt NaN=0
  train: 246,234 rows, 361 days
  val: 62,554 rows, 78 days
  test: 54,587 rows, 78 days


## 4. CS-VAE (64×64 → 64-d) + encode all frames + optical-flow motion features

In [5]:
# ==== SKIPPD_VAE ====
try:
    exec(safe_stage('SKIPPD_VAE', SKIPPD_VAE_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SKIPPD_VAE — continuing to next cell.')


[SKIPPD-VAE] training 12 epochs on 363,375 images (cuda, workers=2) ...


<string>:21: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
<string>:21: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)


  VAE ep 1/12  loss=0.0064  1.0min


<string>:21: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
<string>:21: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)


  VAE ep 2/12  loss=0.0036  2.0min


...

  VAE ep 12/12  loss=0.0031  12.2min
[SKIPPD-VAE] encoding all frames -> latents ...


  encode:   0%|          | 0/710 [00:00<?, ?it/s]<string>:21: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
<string>:21: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  encode: 100%|██████████| 710/710 [01:18<00:00,  8.99it/s]


  latents (363375, 64)
[SKIPPD-VAE] computing optical-flow motion features ...


  flow: 100%|██████████| 363375/363375 [10:40<00:00, 566.94it/s]

  motion features (363375, 4)  (saved motion_norm.npy)


## 5. CTI + write the {splits, extended, latents} contract

In [6]:
# ==== SKIPPD_WRITE ====
try:
    exec(safe_stage('SKIPPD_WRITE', SKIPPD_LATENTS_WRITE_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SKIPPD_WRITE — continuing to next cell.')


[SKIPPD-WRITE] CTI from latent velocity (per-day windowed) ...
  CTI range [0.00e+00, 3.10e+00]
  covariates (363375, 9) (5 time/sky + 4 motion)
[SKIPPD-WRITE] writing splits + latents ...
[SKIPPD-WRITE] writing extended (full PV labels) ...
  extended train: 246,234 rows, 361 days
  extended val: 62,554 rows, 78 days
  extended test: 54,587 rows, 78 days
[SKIPPD-WRITE] contract written. Free image RAM.


## 6. Shared metrics + load tensors + 1-min horizon config

In [7]:
# ==== SHARED ====
try:
    exec(safe_stage('SHARED', SHARED_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SHARED — continuing to next cell.')


Shared code loaded.


In [8]:
# ==== LOAD_DATA ====
try:
    exec(safe_stage('LOAD_DATA', LOAD_DATA_TOLERANT_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] LOAD_DATA — continuing to next cell.')


  CTI normalized: /4.53e-02 (train p90) -> train mean=0.396 p90=1.000 max=10.00

  Covariate dim: 28  (5 original + 15 physics + 8 image features)
  train: Z=(246234, 64), GHI=[0,30], ramps=9917
  val: Z=(62554, 64), GHI=[0,30], ramps=1339
  test: Z=(54587, 64), GHI=[0,29], ramps=1284

8-day image splits: train=246,234 val=62,554 test=54,587
90-day extended:    train=246,234 val=62,554

Z_DIM=64, C_DIM=28
Horizons: [1, 5, 10, 20, 30] min, MC samples: 50, N_EVAL: 2000


In [9]:
# ==== HORIZON_OVERRIDE ====
try:
    exec(safe_stage('HORIZON_OVERRIDE', SKIPPD_HORIZON_OVERRIDE_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] HORIZON_OVERRIDE — continuing to next cell.')


[SKIPPD] horizons=[1, 5, 10, 15, 20, 30] min (1-min cadence), SEQ_LEN=16, PRIMARY_DT=60s, N_EVAL=2000
[SKIPPD] target = rooftop PV (kW); kt = PV / clear-sky-PV envelope


## 6a. Data card + implementation details (reproducibility)

In [10]:
# ==== DATA_CARD ====
try:
    exec(safe_stage('DATA_CARD', DATA_CARD_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] DATA_CARD — continuing to next cell.')


DATA CARD — SKIPP'D splits
split  frames  days  ramp_pct  kt_mean  kt_p10  kt_p90  clear_frac_kt>0.85  cloudy_frac_kt<0.5           months_covered
train  246234   361      4.03    0.720   0.275   0.957               0.463               0.220 1,2,3,5,6,7,8,9,10,11,12
  val   62554    78      2.14    0.739   0.361   0.924               0.447               0.166                  5,6,7,8
 test   54587    78      2.35    0.728   0.321   0.922               0.413               0.174                   8,9,10

  -> saved data_card.csv


In [11]:
# ==== IMPLEMENTATION_DETAILS ====
try:
    exec(safe_stage('IMPLEMENTATION_DETAILS', IMPLEMENTATION_DETAILS_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] IMPLEMENTATION_DETAILS — continuing to next cell.')


IMPLEMENTATION DETAILS / REPRODUCIBILITY
                item                                                                                                                                                                                      value
              python                                                                                                                                                                                    3.12.13
               torch                                                                                                                                                                               2.10.0+cu128
               numpy                                                                                                                                                                                      2.4.6
              pandas                                                                                                           

## 7. ARCHITECTURE A — Closed-form Mixture-of-OU: train + calibrate + evaluate

In [12]:
# ==== CLOSEDFORM_ARCH ====
try:
    exec(safe_stage('CLOSEDFORM_ARCH', MDN_ARCHITECTURE_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CLOSEDFORM_ARCH — continuing to next cell.')


SolarSDE architecture loaded (Temporal Latent Neural SDE: transformer + Mixture-of-OU + persistence-blend + conformal scaling).


In [13]:
# ==== CLOSEDFORM_GLUE ====
try:
    # Keep a named reference to the closed-form class before the rollout
    # architecture overwrites the TemporalLatentSDE alias.
    ClosedFormSDE = TemporalLatentSDE
    print('ClosedFormSDE alias saved.')
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CLOSEDFORM_GLUE — continuing to next cell.')


ClosedFormSDE alias saved.


In [14]:
# ==== CLOSEDFORM_TRAIN ====
try:
    exec(safe_stage('STAGE0_CLOSEDFORM',
         STAGE_0_V2_CODE.replace('EPOCHS = 60', f'EPOCHS = {CLOSEDFORM_EPOCHS}')), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CLOSEDFORM_TRAIN — continuing to next cell.')


STAGE 0: Training Temporal Latent Neural SDE
    Transformer history + Mixture-of-OU (closed-form) +
    persistence-blend floor + post-training conformal calibration

[A] Computing sigma_pers(h) from 90-day extended BMS data (same-day, trimmed) ...
    primary cadence: 60s/step | extended cadence: 60s/step
      h= 1min (lag=1 @ 60s): sigma_pers=0.0331  (kept 99%)
      h= 5min (lag=5 @ 60s): sigma_pers=0.0810  (kept 98%)
      h=10min (lag=10 @ 60s): sigma_pers=0.1056  (kept 98%)
      h=15min (lag=15 @ 60s): sigma_pers=0.1229  (kept 97%)
      h=20min (lag=20 @ 60s): sigma_pers=0.1352  (kept 96%)
      h=30min (lag=30 @ 60s): sigma_pers=0.1559  (kept 95%)
    sigma_pers per horizon (10s steps, same-day, top-1% trimmed): {1: 0.0331, 5: 0.081, 10: 0.1056, 15: 0.1229, 20: 0.1352, 30: 0.1559}
    train pairs: 246,189  val pairs: 62,509  seq_len: 16
    oversampling: 9917 ramp + 61548 high-CTI anchors upweighted


<string>:31: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True


  SDE ep   1/60  train=0.07227  val=0.03378  w_mean=0.563  lr=5.00e-04  1.4min
  SDE ep   5/60  train=0.06136  val=0.03885  w_mean=0.617  lr=4.92e-04  6.8min
  SDE ep  10/60  train=0.05639  val=0.03529  w_mean=0.619  lr=4.67e-04  13.9min
  SDE ep  15/60  train=0.05291  val=0.04286  w_mean=0.618  lr=4.28e-04  21.5min
  SDE ep  20/60  train=0.05120  val=0.04182  w_mean=0.617  lr=3.78e-04  29.4min
  SDE ep  25/60  train=0.04949  val=0.03771  w_mean=0.616  lr=3.18e-04  37.1min
  SDE ep  30/60  train=0.04793  val=0.04574  w_mean=0.615  lr=2.55e-04  44.0min
  SDE ep  35/60  train=0.04692  val=0.03863  w_mean=0.616  lr=1.92e-04  51.0min
  SDE ep  40/60  train=0.04577  val=0.04140  w_mean=0.615  lr=1.33e-04  57.9min
  SDE ep  45/60  train=0.04490  val=0.03981  w_mean=0.614  lr=8.18e-05  64.7min
  SDE ep  50/60  train=0.04406  val=0.04254  w_mean=0.614  lr=4.28e-05  71.6min
  SDE ep  55/60  train=0.04362  val=0.04084  w_mean=0.615  lr=1.83e-05  78.3min
  SDE ep  60/60  train=0.04373  val=0.0416

  h=1min: 100%|██████████| 63/63 [00:00<00:00, 116.06it/s]


    h= 1min  CRPS=0.33  RMSE=1.38  PICP=0.944  PINAW=0.059  ramp_CRPS=4.12


...

    h=15min  CRPS=1.03  RMSE=3.41  PICP=0.923  PINAW=0.176  ramp_CRPS=4.14


  h=20min: 100%|██████████| 63/63 [00:00<00:00, 181.84it/s]


    h=20min  CRPS=1.15  RMSE=3.68  PICP=0.885  PINAW=0.160  ramp_CRPS=4.15


  h=30min: 100%|██████████| 63/63 [00:00<00:00, 176.47it/s]


    h=30min  CRPS=1.32  RMSE=3.95  PICP=0.839  PINAW=0.151  ramp_CRPS=3.96

STAGE 0 COMPLETE — Temporal Latent Neural SDE results
    crps   picp    pinaw     rmse      mae  ramp_crps  horizon_min  horizon_steps  n_eval
0.330990 0.9440 0.058960 1.377310 0.377383   4.121207            1              1    2000
0.752980 0.9270 0.132707 2.758223 0.926947   4.666158            5              5    2000
0.920882 0.9190 0.163623 3.122521 1.180288   4.415043           10             10    2000
1.027253 0.9235 0.176490 3.407959 1.381001   4.135910           15             15    2000
1.147355 0.8855 0.159780 3.681138 1.557765   4.148404           20             20    2000
1.320033 0.8390 0.151161 3.946374 1.763954   3.959252           30             30    2000


In [15]:
# ==== CLOSEDFORM_VERIFY ====
try:
    exec(safe_stage('CLOSEDFORM_VERIFY', POST_STAGE0_V2_VERIFY_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CLOSEDFORM_VERIFY — continuing to next cell.')


[OK] Temporal Latent Neural SDE checkpoint verified (no NaN/Inf).
     conformal_scale = 0.700
     sigma_pers_table = [0.0330733098089695, 0.08101619780063629, 0.10562856495380402, 0.1229119822382927, 0.1351550817489624, 0.1559324860572815]


In [16]:
# ==== STASH_CLOSEDFORM ====
try:
    exec(safe_stage('STASH_CLOSEDFORM', STASH_CLOSEDFORM_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] STASH_CLOSEDFORM — continuing to next cell.')


  stashed closed-form ckpt -> mdn_closedform_best.pt
  stashed closed-form results -> closedform_main_results.csv
  stashed per-horizon preds -> closedform_preds/
[OK] closed-form artifacts stashed; rollout STAGE 0 will train fresh.


## 8. ARCHITECTURE B — Euler-Maruyama latent rollout: train + calibrate + evaluate

In [17]:
# ==== ROLLOUT_ARCH ====
try:
    exec(safe_stage('ROLLOUT_ARCH', ROLLOUT_ARCH_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ROLLOUT_ARCH — continuing to next cell.')


SolarSDE-Rollout architecture loaded (latent neural SDE rolled forward via Euler-Maruyama + decoder; drop-in (pi, mean, std) interface).


In [18]:
# ==== ROLLOUT_TRAIN ====
try:
    exec(safe_stage('STAGE0_ROLLOUT',
         STAGE_0_V2_CODE.replace('EPOCHS = 60', f'EPOCHS = {ROLLOUT_EPOCHS}')), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ROLLOUT_TRAIN — continuing to next cell.')


STAGE 0: Training Temporal Latent Neural SDE
    Transformer history + Mixture-of-OU (closed-form) +
    persistence-blend floor + post-training conformal calibration

[A] Computing sigma_pers(h) from 90-day extended BMS data (same-day, trimmed) ...
    primary cadence: 60s/step | extended cadence: 60s/step
      h= 1min (lag=1 @ 60s): sigma_pers=0.0331  (kept 99%)
      h= 5min (lag=5 @ 60s): sigma_pers=0.0810  (kept 98%)
      h=10min (lag=10 @ 60s): sigma_pers=0.1056  (kept 98%)
      h=15min (lag=15 @ 60s): sigma_pers=0.1229  (kept 97%)
      h=20min (lag=20 @ 60s): sigma_pers=0.1352  (kept 96%)
      h=30min (lag=30 @ 60s): sigma_pers=0.1559  (kept 95%)
    sigma_pers per horizon (10s steps, same-day, top-1% trimmed): {1: 0.0331, 5: 0.081, 10: 0.1056, 15: 0.1229, 20: 0.1352, 30: 0.1559}
    train pairs: 246,189  val pairs: 62,509  seq_len: 16
    oversampling: 9917 ramp + 61548 high-CTI anchors upweighted


<string>:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True


  SDE ep   1/35  train=0.07321  val=0.03626  w_mean=0.581  lr=4.99e-04  4.8min
  SDE ep   5/35  train=0.06126  val=0.03409  w_mean=0.622  lr=4.76e-04  24.4min
  SDE ep  10/35  train=0.05481  val=0.03841  w_mean=0.625  lr=4.08e-04  48.4min
  SDE ep  15/35  train=0.05091  val=0.03893  w_mean=0.624  lr=3.10e-04  72.9min
  SDE ep  20/35  train=0.04757  val=0.03911  w_mean=0.624  lr=2.00e-04  97.4min
  SDE ep  25/35  train=0.04478  val=0.04014  w_mean=0.624  lr=1.02e-04  121.5min
  SDE ep  30/35  train=0.04298  val=0.04138  w_mean=0.624  lr=3.43e-05  146.4min
  SDE ep  35/35  train=0.04242  val=0.04142  w_mean=0.625  lr=1.00e-05  170.8min
  SDE done. Best val CRPS = 0.03360. Time: 170.8 min

[D] Mondrian conformal calibration (direct coverage targeting) on val ...
    CTI quartile cuts (q25, q50, q75): [0.0065100002102553844, 0.016359999775886536, 0.07576999813318253]
    base conformal scales (coverage-targeted): {1: 0.7, 5: 0.7, 10: 0.7, 15: 0.8, 20: 0.8, 30: 0.9}
    CTI-quartile multipl

  h=1min: 100%|██████████| 63/63 [00:00<00:00, 148.49it/s]


    h= 1min  CRPS=0.35  RMSE=1.37  PICP=0.959  PINAW=0.075  ramp_CRPS=3.82


...

    h=15min  CRPS=1.01  RMSE=3.22  PICP=0.895  PINAW=0.166  ramp_CRPS=3.67


  h=20min: 100%|██████████| 63/63 [00:01<00:00, 40.19it/s]


    h=20min  CRPS=1.08  RMSE=3.37  PICP=0.883  PINAW=0.166  ramp_CRPS=3.31


  h=30min: 100%|██████████| 63/63 [00:02<00:00, 29.81it/s]


    h=30min  CRPS=1.16  RMSE=3.49  PICP=0.882  PINAW=0.169  ramp_CRPS=3.38

STAGE 0 COMPLETE — Temporal Latent Neural SDE results
    crps   picp    pinaw     rmse      mae  ramp_crps  horizon_min  horizon_steps  n_eval
0.345705 0.9585 0.075323 1.371208 0.428951   3.815191            1              1    2000
0.742762 0.9410 0.145599 2.675800 0.967877   4.350980            5              5    2000
0.894291 0.9225 0.171352 2.999007 1.246805   3.812752           10             10    2000
1.007213 0.8955 0.165768 3.220249 1.408006   3.667952           15             15    2000
1.076397 0.8830 0.165579 3.372760 1.525237   3.312602           20             20    2000
1.158678 0.8825 0.169046 3.488954 1.652493   3.380564           30             30    2000


In [19]:
# ==== ROLLOUT_VERIFY ====
try:
    exec(safe_stage('ROLLOUT_VERIFY', POST_STAGE0_V2_VERIFY_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ROLLOUT_VERIFY — continuing to next cell.')


[OK] Temporal Latent Neural SDE checkpoint verified (no NaN/Inf).
     conformal_scale = 0.700
     sigma_pers_table = [0.0330733098089695, 0.08101619780063629, 0.10562856495380402, 0.1229119822382927, 0.1351550817489624, 0.1559324860572815]


In [20]:
# ==== STASH_ROLLOUT ====
try:
    exec(safe_stage('STASH_ROLLOUT', STASH_ROLLOUT_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] STASH_ROLLOUT — continuing to next cell.')


  stashed rollout ckpt -> mdn_rollout_best.pt
  stashed rollout results -> rollout_main_results.csv
  stashed per-horizon preds -> rollout_preds/
[OK] rollout artifacts stashed.


## 9. Champion selection (on VALIDATION) — closed-form vs rollout vs ensemble

In [21]:
# ==== CHAMPION_SELECT ====
try:
    exec(safe_stage('CHAMPION_SELECT', CHAMPION_SELECT_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CHAMPION_SELECT — continuing to next cell.')


CHAMPION SELECTION on VALIDATION (h=15min, CRPS)
  closed-form ckpt: True | rollout ckpt: True


<string>:31: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
<string>:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True


  ensemble weight optimized on val: w(closed-form)=0.10
  val CRPS @ h=15min  ensemble     = 0.7602
  val CRPS @ h=15min  rollout      = 0.7620
  val CRPS @ h=15min  closedform   = 0.7850

  CHAMPION (selected on validation): ensemble
  downstream suite runs on: rollout (canonical mdn_v2_best.pt restored)
  ensemble weight for benchmark: w(closed-form)=0.10


## 10. SkyGPT EXACT BENCHMARK — all variants, identical cloudy test, full 1–30 min band

In [22]:
# ==== SKYGPT_TRIPLE_BENCHMARK ====
try:
    exec(safe_stage('SKYGPT_TRIPLE_BENCHMARK', SKYGPT_TRIPLE_BENCHMARK_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SKYGPT_TRIPLE_BENCHMARK — continuing to next cell.')


  downloading test_set_2019nov_dec.hdf5 ...
  downloading times_curr_test_2019nov_dec.npy ...
SkyGPT EXACT BENCHMARK — all variants (2582 windows, 5 cloudy days)
  encoding log frames ...
  computing motion features ...


<string>:31: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
<string>:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True


  variants available: ['closedform', 'rollout'] + ensemble
  h= 1min  closedform=1.234  rollout=1.260  ensemble=1.255  smart-pers=1.241  [best: closedform]
  h= 5min  closedform=2.194  rollout=2.257  ensemble=2.249  smart-pers=2.150  [best: closedform]
  h=10min  closedform=2.794  rollout=2.924  ensemble=2.899  smart-pers=2.748  [best: closedform]
  h=15min  closedform=3.179  rollout=3.366  ensemble=3.334  smart-pers=3.105  [best: closedform]
  h=20min  closedform=3.496  rollout=3.647  ensemble=3.600  smart-pers=3.290  [best: closedform]
  h=30min  closedform=4.042  rollout=4.049  ensemble=4.013  smart-pers=3.471  [best: ensemble]

HEAD-TO-HEAD at h=15min (SkyGPT's identical cloudy test set):
                       method  crps_kW  winkler  skill_vs_smartpers_%
    SkyGPT->U-Net (published)    2.810    26.70                  23.0
   SolarSDE-closedform (ours)    3.179    31.17                  -2.4
           SUNSET (published)    3.310    56.95                   9.8
     SolarSDE-ense

## 11. Ablations (champion-matched: closed-form or rollout native)

In [23]:
# ==== ABLATIONS ====
try:
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    _abl = ABLATIONS_V2_CODE if globals().get('CHAMPION_SINGLE', 'closedform') == 'closedform' \
           else ABLATIONS_ROLLOUT_CODE
    exec(safe_stage('ABLATIONS', _abl), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ABLATIONS — continuing to next cell.')


STAGE B (rollout): Ablations of RolloutLatentSDE

  A1: full rollout model


<string>:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True


  A2: no CTI-gated diffusion
  A4: no persistence-blend (w forced to 1)
  A5: deterministic rollout (no diffusion noise -> Neural ODE)
  A7: no covariates

Ablation summary @ h=10min:
         ablation  crps  picp  pinaw  rmse
          A1_full 1.961 0.859  0.294 4.703
        A2_no_cti 1.976 0.829  0.269 4.717
A4_no_persistence 2.141 0.672  0.254 4.831
    A5_no_sde_ODE 1.940 0.862  0.291 4.648
 A7_no_covariates 2.128 0.861  0.324 4.845
  -> saved /kaggle/working/final_run/outputs/results/ablation_results.csv


## 12. Leave-one-month-out cross-validation

In [24]:
# ==== CROSS_VALIDATION ====
try:
    exec(safe_stage('CROSS_VALIDATION', CROSS_VALIDATION_V2_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CROSS_VALIDATION — continuing to next cell.')


CROSS-VALIDATION (month-grouped (6), 6 folds, 15 epochs/fold)


<string>:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True


  fold 0: trained on 294,911 anchors, tested on 68,374
  fold 1: trained on 313,597 anchors, tested on 49,643
  fold 2: trained on 284,718 anchors, tested on 78,522
  fold 3: trained on 322,404 anchors, tested on 40,836
  fold 4: trained on 324,285 anchors, tested on 38,955
  fold 5: trained on 276,510 anchors, tested on 86,775

Cross-validation summary (mean +/- std across folds):
  h= 1min  CRPS=0.211+/-0.088  PICP=0.965+/-0.026  (6 folds)
  h= 5min  CRPS=0.427+/-0.200  PICP=0.952+/-0.033  (6 folds)
  h=10min  CRPS=0.526+/-0.256  PICP=0.946+/-0.032  (6 folds)
  h=15min  CRPS=0.589+/-0.289  PICP=0.932+/-0.032  (6 folds)
  h=20min  CRPS=0.646+/-0.319  PICP=0.887+/-0.062  (6 folds)
  h=30min  CRPS=0.732+/-0.369  PICP=0.798+/-0.132  (6 folds)
  -> saved cross_validation_results.csv, cross_validation_summary.csv


## 13. Sampling efficiency + computational cost

In [25]:
# ==== SAMPLING_EFFICIENCY ====
try:
    exec(safe_stage('SAMPLING_EFFICIENCY', SAMPLING_EFFICIENCY_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SAMPLING_EFFICIENCY — continuing to next cell.')


<string>:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True


  N=  10:  CRPS=1.9339  PICP=0.7170
  N=  25:  CRPS=1.8477  PICP=0.8050
  N=  50:  CRPS=1.8172  PICP=0.8400
  N= 100:  CRPS=1.8024  PICP=0.8490
  N= 200:  CRPS=1.7989  PICP=0.8620
SAMPLING EFFICIENCY (h=15min)
 n_samples   crps  picp
        10 1.9339 0.717
        25 1.8477 0.805
        50 1.8172 0.840
       100 1.8024 0.849
       200 1.7989 0.862
  CRPS gain from N=50->200: 1.01%  (N=50 is near-converged; default is a good speed/quality tradeoff)
  -> saved sampling_efficiency.csv


In [26]:
# ==== COMPUTATIONAL_COST ====
try:
    exec(safe_stage('COMPUTATIONAL_COST', COMPUTATIONAL_COST_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] COMPUTATIONAL_COST — continuing to next cell.')


<string>:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True


COMPUTATIONAL COST
total_params                       617604
trainable_params                   617604
model_size_MB                        2.47
sde_train_minutes                     n/a
inference_latency_ms_per_forecast  12.761
forecasts_per_second                 78.4
device                               cuda
mc_samples                             50

  Model is 0.62M params, 2.5 MB — runs in 12.8 ms/forecast on cuda (real-time capable for 1-min nowcasting).
  -> saved computational_cost.csv


## 14. Stratified eval + Diebold-Mariano significance

In [27]:
# ==== STRATIFIED ====
try:
    exec(safe_stage('STRATIFIED', STRATIFIED_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] STRATIFIED — continuing to next cell.')


STAGE C2: Stratified evaluation (where does SolarSDE actually win?)

Stratified analysis at h=10min:
                          subset  n_samples  solarsde_crps  persistence_crps    delta   winner
                 All test points       2000       0.894291          2.206326 1.312035 SolarSDE
               CTI Q1 (clearest)        499       0.192615          1.434709 1.242094 SolarSDE
                          CTI Q2        498       0.156865          1.452667 1.295802 SolarSDE
                          CTI Q3        498       0.275868          1.516044 1.240176 SolarSDE
         CTI Q4 (most turbulent)        499       2.958854          4.439384 1.480529 SolarSDE
    CTI top 10% (most turbulent)        200       3.457545          4.855909 1.398364 SolarSDE
               Clear (kt > 0.85)       1061       0.673486          1.999878 1.326393 SolarSDE
Partial cloud (0.5 < kt <= 0.85)        549       0.829548          2.014927 1.185379 SolarSDE
              Cloudy (kt <= 0.5)        390 

## 15. PIT / reliability + bootstrap CIs

In [28]:
# ==== PIT_RELIABILITY ====
try:
    exec(safe_stage('PIT_RELIABILITY', PIT_RELIABILITY_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] PIT_RELIABILITY — continuing to next cell.')



PIT + reliability + sharpness saved to FIGURES_DIR / RESULTS_DIR.
      model  horizon_min  sharpness_90     crps
   SolarSDE           10      5.016509 0.894291
Persistence           10     14.580842 2.197692


In [29]:
# ==== BOOTSTRAP_CIS ====
try:
    exec(safe_stage('BOOTSTRAP_CIS', BOOTSTRAP_CIS_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] BOOTSTRAP_CIS — continuing to next cell.')


  Loaded test_predictions_h10min.npz: keys = ['y_true', 'y_samples', 'is_ramp', 'truths', 'preds']

  SolarSDE @ 10min:  CRPS = 0.89  [0.81, 0.98]  (B=1000)
                     RMSE = 2.94  [2.71, 3.15]
                     MAE  = 1.23  [1.12, 1.34]

Bootstrap CIs at all horizons:
  h= 1min  CRPS=  0.35 [ 0.30,  0.39]  RMSE=  1.37 [ 1.20,  1.55]  skill=+nan%
  h= 5min  CRPS=  0.74 [ 0.67,  0.82]  RMSE=  2.61 [ 2.37,  2.84]  skill=+nan%
  h=10min  CRPS=  0.89 [ 0.81,  0.98]  RMSE=  2.94 [ 2.71,  3.15]  skill=+nan%
  h=15min  CRPS=  1.01 [ 0.91,  1.09]  RMSE=  3.16 [ 2.92,  3.37]  skill=+nan%
  h=20min  CRPS=  1.08 [ 0.98,  1.17]  RMSE=  3.30 [ 3.08,  3.52]  skill=+nan%
  h=30min  CRPS=  1.16 [ 1.06,  1.25]  RMSE=  3.44 [ 3.21,  3.65]  skill=+nan%
  -> saved bootstrap_cis_all_horizons.csv


## 16. Ramp AUROC + CTI physical validation

In [30]:
# ==== RAMP_AUROC ====
try:
    exec(safe_stage('RAMP_AUROC', RAMP_AUROC_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] RAMP_AUROC — continuing to next cell.')


Ramp events in test: 1284 (2.4% of timestamps)
  h= 1min  Ramp AUROC (PI-width)  = 0.899  (n_ramp=106)
  h= 5min  Ramp AUROC (PI-width)  = 0.905  (n_ramp=106)
  h=10min  Ramp AUROC (PI-width)  = 0.904  (n_ramp=106)
  h=15min  Ramp AUROC (PI-width)  = 0.905  (n_ramp=106)
  h=20min  Ramp AUROC (PI-width)  = 0.912  (n_ramp=106)
  h=30min  Ramp AUROC (PI-width)  = 0.913  (n_ramp=106)

CTI around ramp events (1284 events):
  mean CTI in 5 steps BEFORE ramp: 1.3160
  mean CTI AT ramp (t=0):                   1.3975
  mean CTI in 30 steps AFTER ramp:   1.2829
  CTI rise from before to at-ramp: +6.2%


In [31]:
# ==== CTI_VALIDATION ====
try:
    exec(safe_stage('CTI_VALIDATION', CTI_VALIDATION_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CTI_VALIDATION — continuing to next cell.')


## 17. Multi-level reliability

In [32]:
# ==== RELIABILITY_LEVELS ====
try:
    exec(safe_stage('RELIABILITY_LEVELS', RELIABILITY_LEVELS_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] RELIABILITY_LEVELS — continuing to next cell.')


RELIABILITY ACROSS CONFIDENCE LEVELS
Empirical coverage (rows=horizon_min, cols=nominal level):
nominal       0.50   0.60   0.70   0.80   0.90   0.95
horizon_min                                          
1            0.778  0.862  0.908  0.934  0.958  0.972
5            0.636  0.774  0.863  0.908  0.941  0.959
10           0.472  0.639  0.774  0.864  0.922  0.948
15           0.379  0.520  0.673  0.800  0.896  0.934
20           0.376  0.491  0.627  0.754  0.883  0.933
30           0.427  0.566  0.690  0.796  0.882  0.922

Mean abs calibration error per horizon (lower=better):
horizon_min
1     0.1605
5     0.1051
10    0.0382
15    0.0417
20    0.0642
30    0.0278
  overall ECE = 0.0729
  -> saved reliability_levels.csv


## 18. Holm-Bonferroni + CAISO economics + sensitivity

In [33]:
# ==== HOLM_BONFERRONI ====
try:
    exec(safe_stage('HOLM_BONFERRONI', HOLM_BONFERRONI_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] HOLM_BONFERRONI — continuing to next cell.')


[INFO] No DM p-value columns found in stratified_results.csv — skipping Holm-Bonferroni.


In [34]:
# ==== ECONOMIC_CAISO ====
try:
    exec(safe_stage('ECONOMIC_CAISO', ECONOMIC_CAISO_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ECONOMIC_CAISO — continuing to next cell.')



Economic value (CAISO reserve simulation, h=10min, 1 GW solar plant):
            model  horizon_min  mean_reserve_held_pu  mean_shortfall_pu  annual_cost_per_GW_USD
         SolarSDE           10              0.524786           0.002678            2.533171e+08
      Persistence           10              0.702676           0.007710            3.753157e+08
Smart-Persistence           10              0.615891           0.011936            3.743234e+08

SolarSDE annual reserve savings vs persistence: $121,998,646 per GW per year
Equivalent for a 10 GW solar deployment:        $1,219,986,461 per year


In [35]:
# ==== ECONOMIC_SENSITIVITY ====
try:
    exec(safe_stage('ECONOMIC_SENSITIVITY', ECONOMIC_SENSITIVITY_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ECONOMIC_SENSITIVITY — continuing to next cell.')


ECONOMIC VALUE — SENSITIVITY
Savings vs persistence by horizon ($50 reserve / $1000 penalty / 1 GW):
 horizon_min  sde_cost_USD_per_GW_yr  persistence_cost_USD_per_GW_yr  savings_USD_per_GW_yr
           1               219917424                       391019389              171101965
           5               248767488                       388648563              139881075
          10               253317072                       390081830              136764758
          15               250795616                       392275807              141480191
          20               243848992                       393476970              149627978
          30               245462112                       397760003              152297891

Price-grid sweep at h=10min (savings $/GW/yr):
penalty_$per_MWh      500        1000       2000
reserve_$per_MWh                                
30                69986585  131972797  255945188
50                75320151  137306363  261278738
80         

## 19. Baselines (persistence, smart-persistence, LSTM, MC-Dropout, CSDI)

In [36]:
# ==== BASELINES ====
try:
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    exec(safe_stage('BASELINES', BASELINES_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] BASELINES — continuing to next cell.')


STAGE A: Training baselines

[A1] Persistence
  h=1min: CRPS=0.50 RMSE=1.41 PICP=0.940
  h=5min: CRPS=1.13 RMSE=2.91 PICP=0.920
  h=10min: CRPS=1.43 RMSE=3.45 PICP=0.907
  h=15min: CRPS=1.72 RMSE=3.91 PICP=0.898
  h=20min: CRPS=1.93 RMSE=4.20 PICP=0.887
  h=30min: CRPS=2.17 RMSE=4.46 PICP=0.896

[A2] Smart Persistence
  h=1min: CRPS=0.50 RMSE=1.41 PICP=0.939
  h=5min: CRPS=1.11 RMSE=2.91 PICP=0.918
  h=10min: CRPS=1.38 RMSE=3.42 PICP=0.900
  h=15min: CRPS=1.61 RMSE=3.85 PICP=0.889
  h=20min: CRPS=1.77 RMSE=4.11 PICP=0.875
  h=30min: CRPS=1.84 RMSE=4.20 PICP=0.869

[A3/A4] Building LSTM sequence tensors (extended 90-day BMS)
    target scale set to 35.5 (from train max 29.6) — PV-aware, prevents residual underflow
  Seq shapes: train=torch.Size([40993, 16, 2])  val=torch.Size([10380, 16, 2])  test=torch.Size([54541, 16, 2])  (LSTM target = persistence-residual / 35, CSDI uses raw GHI)

[A3] LSTM deterministic (40 epochs, persistence-residual target)
    lstm_det ep 1/40 tr=0.0141 val=0.

## 20. Analysis figures + LaTeX tables

In [37]:
# ==== ANALYSIS ====
try:
    exec(safe_stage('ANALYSIS', ANALYSIS_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ANALYSIS — continuing to next cell.')


STAGE D: Analysis + figures

[D1] CTI analysis
  CTI vs GHI-var Spearman rho=0.296, p=0.00e+00, N=54401
  Q1: CTI=0.0047, CRPS=0.19, N=499
  Q2: CTI=0.0129, CRPS=0.16, N=498
  Q3: CTI=0.0668, CRPS=0.28, N=498
  Q4: CTI=1.1613, CRPS=2.96, N=499
  Clear: CTI=0.0658, GHI=13.2±7.3, N=48763
  Thin Cloud: CTI=1.0534, GHI=11.3±7.1, N=4222
  Broken Cloud: CTI=2.8564, GHI=10.5±6.3, N=1234
  Overcast: CTI=6.7678, GHI=10.7±5.7, N=212

[D2] Economic value
  SolarSDE annual: $3.53M/GW
  Persistence:     $5.12M/GW
  Savings:         $1.59M/GW/yr  (31.1% reduction)

[D3] Generating figures
  saved fig2_crps_vs_horizon.pdf
  saved fig3a_cti_scatter.pdf
  saved fig3b_crps_by_cti.pdf
  saved fig5_reliability.pdf
  saved fig6_economic_value.pdf
  saved fig_pit_histogram.pdf

All figures saved to: /kaggle/working/final_run/outputs/figures

=== PAPER TABLE 1 — main results at h=10min ===
            model  horizon_min     crps     rmse      mae   picp    pinaw  ramp_crps  skill_vs_persistence
         Sola

In [38]:
# ==== LATEX_TABLES ====
try:
    exec(safe_stage('LATEX_TABLES', LATEX_TABLES_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] LATEX_TABLES — continuing to next cell.')



LaTeX tables saved:
  /kaggle/working/final_run/outputs/results/table1_main_crps.tex  (True)
  /kaggle/working/final_run/outputs/results/table2_ablations.tex  (True)
  /kaggle/working/final_run/outputs/results/table3_computational.tex  (False)


## Final — Zip the complete paper package

In [39]:
# ==== ZIP ====
try:
    exec(safe_stage('ZIP', ZIP_DOWNLOAD_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ZIP — continuing to next cell.')


Zipping /kaggle/working/final_run/outputs -> solarsde_outputs.zip ...
  Archive size: 141.8 MB

ALL STAGES COMPLETE
  splits/: 3 files, 7.3 MB
  extended/: 3 files, 9.2 MB
  checkpoints/: 10 files, 20.9 MB
  latents/: 27 files, 139.9 MB
  results/: 58 files, 8.2 MB
  figures/: 10 files, 0.3 MB

Cleaning intermediate files (MINIMAL_OUTPUT=True) ...
  removed outputs/ (contents archived in zip)
  removed work/ (raw downloads)

Final /kaggle/working/ contents (4 entries):
  .virtual_documents/
  final_run/
  solarsde_outputs.zip  (141.8 MB)
  solarsde_outputs_summary.csv  (0.0 MB)

Download the zip from the Output tab on the right sidebar.
Or 'Save Version' to commit /kaggle/working/ as a Kaggle Dataset for the next notebook.
